# Problem 567- Reciprocal Games I

Tom has built a random generator that is connected to a row of $n$ light bulbs. Whenever the random generator is activated each of the $n$ lights is turned on with the probability of $\frac 1 2$, independently of its former state or the state of the other light bulbs.

While discussing with his friend Jerry how to use his generator, they invent two different games, they call the **reciprocal games**:
Both games consist of $n$ turns. Each turn is started by choosing a number $k$ randomly between (and including) $1$ and $n$, with equal probability of $\frac 1 n$ for each number, while the possible win for that turn is the reciprocal of $k$, that is $\frac 1 k$.

In game A, Tom activates his random generator once in each turn. If the number of lights turned on is the same as the previously chosen number $k$, Jerry wins and gets $\frac 1 k$, otherwise he will receive nothing for that turn. Jerry's expected win after playing the total game A consisting of $n$ turns is called $J_A(n)$. For example $J_A(6)=0.39505208$, rounded to $8$ decimal places.

For each turn in game B, after $k$ has been randomly selected, Tom keeps reactivating his random generator until exactly $k$ lights are turned on. After that Jerry takes over and reactivates the random generator until he, too, has generated a pattern with exactly $k$ lights turned on. If this pattern is identical to Tom's last pattern, Jerry wins and gets $\frac 1 k$, otherwise he will receive nothing. Jerry's expected win after the total game B consisting of $n$ turns is called $J_B(n)$. For example $J_B(6)=0.43333333$, rounded to $8$ decimal places.

Let $\displaystyle S(m)=\sum_{n=1}^m (J_A(n)+J_B(n))$. For example $S(6)=7.58932292$, rounded to $8$ decimal places.

Find $S(123456789)$, rounded to $8$ decimal places.

## Solution.

**Definitions & Closed Forms**

* $J_A(n) = \frac{1}{2^n} \sum_{k=1}^n \frac{1}{k} \binom{n}{k} = \frac{1}{2^n} \sum_{k=1}^n \frac{2^k - 1}{k}$
* $J_B(n) = \sum_{k=1}^n \frac{1}{k \binom{n}{k}} = \frac{1}{2^n} \sum_{k=1}^n \frac{2^k}{k}$
* $J_A(n) + J_B(n) = \frac{1}{2^n} \sum_{k=1}^n \frac{2^{k+1} - 1}{k}$

---

**Integral Identities**

* **Reciprocal $k$ Integral:**
  $$\frac{1}{k} = \int_0^1 x^{k-1} dx \implies \sum_{k=1}^n \frac{1}{k}\binom{n}{k} = \int_0^1 \frac{(1+x)^n - 1}{x} dx$$

* **Beta Integral for Reciprocal Binomial Coefficient:**
  $$\frac{1}{k \binom{n}{k}} = \int_0^1 x^{k-1}(1-x)^{n-k} dx \implies \sum_{k=1}^n \frac{1}{k \binom{n}{k}} = \int_0^1 \frac{x^n - (1-x)^n}{2x - 1} dx$$

---

**Exact Outer Sum $S(m)$**

$$S(m) = \sum_{n=1}^m (J_A(n) + J_B(n)) = (4 + 2^{-m}) H_m - 2^{1-m} A_m - 2 B_m$$

* $H_m = \sum_{k=1}^m \frac{1}{k}$
* $A_m = \sum_{k=1}^m \frac{2^k}{k}$
* $B_m = \sum_{k=1}^m \frac{1}{k 2^k}$

---

**Approximations & Tail Expansions**

* **Euler-Maclaurin Tail Expansion for $H_m$ ($N < m$):**
  $$H_m - H_N \approx \ln\left(\frac{m}{N}\right) - \frac{1}{2N} + \frac{1}{2m} + \frac{1}{12N^2} - \frac{1}{12m^2} - \frac{1}{120N^4} + \frac{1}{120m^4}$$

* **Exponential Term Expansion:**
  $$2^{1-m} A_m \approx \frac{4}{m} + \frac{4}{m^2} + \frac{12}{m^3}$$

* **Constant Limit for $B_m$:**
  $$B_m \approx \ln(2)$$

* **Full Asymptotic Expansion:**
  $$S(m) \approx 4\ln(m) + 4\gamma - 2\ln(2) - \frac{2}{m} - \frac{13}{3m^2} \quad (\gamma \approx 0.5772156649)$$

In [2]:
from functools import cache
from math import comb

In [5]:
@cache
def J_A(n):
    return (1/2)**n * sum(comb(n, k) * 1/k for k in range(1, n+1))

In [10]:
def J_B(n):
    return sum(comb(n, k)**(-1) * 1/k for k in range(1, n+1))

In [11]:
J_A(6), J_B(6)

(0.39505208333333336, 0.43333333333333335)

In [12]:
def S(m):
    return sum(J_A(n) + J_B(n) for n in range(1, m+1))

In [34]:
def S_fast(m, N=10000):
    H_N = sum(1.0 / k for k in range(1, N + 1))
    B_N = sum(1.0 / (k * (2.0**k)) if k < 1024 else 0.0 for k in range(1, N + 1))

    ln_ratio = math.log(m / N)
    tail_H = (
        ln_ratio 
        - 1.0 / (2 * N) + 1.0 / (2 * m)
        + 1.0 / (12 * N**2) - 1.0 / (12 * m**2)
        - 1.0 / (120 * N**4) + 1.0 / (120 * m**4)
    )
    H_m = H_N + tail_H

    term_A = 4.0 / m + 4.0 / (m**2) + 12.0 / (m**3)

    result = 4.0 * H_m - 2.0 * B_N - term_A
    return round(result, 8)

In [35]:
S_fast(123456789)

75.44817535